# 01 — Global Moran's I Analysis

Computes global Moran's I for medical and sports facility distributions at province and city levels (Section 4.2).

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely import wkt
from esda.moran import Moran
from libpysal.weights import KNN

# ========================================
# 1) Load panel data
# ========================================
csv_path = "jiangxi_causal_forest_panel_1km_with_city.csv"
df = pd.read_csv(csv_path)
df["geometry"] = df["geometry"].apply(wkt.loads)
gdf = gpd.GeoDataFrame(df, geometry="geometry")

# ========================================
# 2) Compute Moran's I by year × city × facility type
# ========================================
facility_types = ["medical_cnt", "sports_cnt"]
results = []

for year in sorted(gdf["year"].unique()):
    for city in sorted(gdf["city"].unique()):
        subset = gdf[(gdf["year"] == year) & (gdf["city"] == city)].copy()
        subset = subset[subset.geometry.notnull()]
        subset = subset.drop_duplicates(subset="geometry")
        subset = subset.reset_index(drop=True)
        
        if subset.shape[0] < 10:
            continue
            
        for var in facility_types:
            if subset[var].sum() == 0 or subset[var].std() == 0:
                continue
            try:
                k = min(8, len(subset) - 1)
                w = KNN.from_dataframe(subset, k=k)
                w.transform = "r"
                x = subset[var].values.astype(float)
                mi = Moran(x, w)
                results.append({
                    "year": year, "city": city, "facility_type": var,
                    "moran_I": mi.I, "p_value": mi.p_sim,
                    "z_score": mi.z_sim, "count": subset.shape[0]
                })
            except Exception as e:
                print(f"Skipped {year} {city} {var}: {e}")

result_df = pd.DataFrame(results).sort_values(["year", "city", "facility_type"])
result_df.to_csv("moran_results_by_year_city.csv", index=False, encoding="utf-8-sig")
print(f"Saved {len(result_df)} records to moran_results_by_year_city.csv")
print(result_df.head(10))

## Plot: Annual Moran's I Trend (Province + Cities)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Times New Roman"

df = pd.read_csv("moran_results_by_year_city.csv")

# Filter for medical facilities
med = df[df["facility_type"] == "medical_cnt"].copy()

# Province-level average per year
prov = med.groupby("year")["moran_I"].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 6))

# Province average (bold line)
ax.plot(prov["year"], prov["moran_I"], linewidth=2.5, marker="o",
        color="black", label="Province Average")

# City-level lines
for city, sub in med.groupby("city"):
    sub = sub.sort_values("year")
    ax.plot(sub["year"], sub["moran_I"], linewidth=1.0, alpha=0.6, label=city)

ax.axhline(0, linestyle="--", linewidth=1, color="gray", alpha=0.7)
ax.set_xlabel("Year", fontsize=14)
ax.set_ylabel("Global Moran's I (Medical Facilities)", fontsize=14)
ax.set_title("Annual Global Moran's I in Jiangxi Province (2012-2023)", fontsize=16, fontweight="bold")
ax.legend(fontsize=9, ncol=4, frameon=False)
ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)
plt.tight_layout()
plt.savefig("moran_medical_trend.png", dpi=300)
plt.show()
print("Saved: moran_medical_trend.png")
